# 🍏 Setting Up AllenNLP on Apple Silicon (M1/M2/M3/M4 Macs)

**The Problem:** AllenNLP is a legacy library that relies on older dependencies (`transformers`, `tokenizers`, and `spaCy`). These older versions do not have pre-built installation files for modern Mac chips (ARM64). If you try to install them normally, your Mac will attempt to build them from scratch using Rust and C-compilers, which will fail and throw a `cargo rustc` error.

**The Solution:** We will force Conda to create an **Intel (x86_64)** environment. Your Mac will use Rosetta 2 to seamlessly translate it in the background, allowing us to download the pre-compiled Intel files and bypass the compiler errors entirely.

---

### ⚠️ Prerequisites
* You must have **Miniconda** or **Anaconda** installed.
* **Important:** Run Steps 1-5 in your Mac's **Terminal application**, *not* inside a Jupyter code cell!

---

### Step 1: Create the Intel-Emulated Environment
We use a special environment variable (`CONDA_SUBDIR`) to trick Conda into thinking it's running on an older Intel Mac. We are using **Python 3.10** as it is the most stable version for these legacy libraries.

`CONDA_SUBDIR=osx-64 conda create -n allennlp_intel python=3.10 -y`

### Step 2: Activate and Lock the Environment
Activate the environment, and then lock its architecture to Intel so that future package installations don't accidentally revert to ARM64.

`conda activate allennlp_intel`
`conda config --env --set subdir osx-64`

### Step 3: Install the Core AllenNLP Packages
Now, you can install AllenNLP. Because the environment is emulating Intel, it will successfully download the pre-built files instead of throwing the Rust compiler error.

`pip install allennlp allennlp-models`

### Step 4: The "NumPy 2.0" Fix
By default, the previous step installs the newest version of NumPy (2.x). However, AllenNLP relies on an older version of `spaCy` that expects NumPy 1.x. If you skip this step, you will get a `ValueError: numpy.dtype size changed` when running your code. Force the environment to downgrade NumPy:

`pip install "numpy<2"`

### Step 5: Hook It Up to Jupyter Notebook
To use this specific environment inside a Jupyter Notebook, you need to register it as a "Kernel."

`pip install ipykernel`

---

### 🛑 CRITICAL JUPYTER NOTE
When you open this Jupyter Notebook, make sure you select **"Python (AllenNLP Intel)"** from the Kernel menu at the top right. 

If you run your code and *still* get the NumPy memory error, go to the top menu and click **Kernel -> Restart Kernel**. Jupyter sometimes holds the old, broken NumPy in memory until you force it to restart.

1- CONDA_SUBDIR=osx-64 conda create -n allennlp_intel python=3.10 -y

2- conda activate allennlp_intel

3- conda config --env --set subdir osx-64

4- pip install allennlp allennlp-models

5- pip install "numpy<2"

6- pip install ipykernel


In [1]:
from allennlp.predictors.predictor import Predictor
import allennlp_models.tagging

print("Loading AllenNLP Model...")
predictor = Predictor.from_path("https://storage.googleapis.com/allennlp-public-models/bert-base-srl-2020.11.19.tar.gz")
print("Model loaded successfully!\n")

sentence = "The software engineer updated the legacy code."
prediction = predictor.predict(sentence=sentence)

for verb_data in prediction['verbs']:
    print(f"Predicate: {verb_data['verb']}")
    print(f"Roles: {verb_data['description']}")

/opt/miniconda3/envs/allennlp_intel/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/allennlp_intel/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Loading AllenNLP Model...


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Model loaded successfully!

Predicate: updated
Roles: [ARG0: The software engineer] [V: updated] [ARG1: the legacy code] .
